# BatchGeoCache — Batch Geocoding Pipeline (Colab)

Mirrors the prior Calgary-specific prototype's notebook flow, generalized and de-Drive'd per
`PROGRESS_05.md` Section 1: **upload → normalize/geocode in packages → inspect/resume →
combine → zip → download → optional reset**, installed straight from GitHub via `git clone`
(no Google Drive mount, no PyPI).

> ⚠️ **Assumptions flagged inline below (marked `# CONFIRM:`)** are best-guess based on the
> module/function names already fixed in `PROGRESS_05.md`'s Decision Log and Step 4/5 entries,
> since this session doesn't have the actual migrated source in context. Step 7
> ("Test/debug notebook end-to-end") is exactly where these get corrected against the real
> signatures — flagging them now rather than guessing silently.

## 1. Install BatchGeoCache (git clone — no Drive, no PyPI)

In [ ]:
# CONFIRM: replace with the actual BatchGeoCache repo URL
REPO_URL = "https://github.com/<YOUR_GITHUB_USERNAME>/BatchGeoCache.git"

!git clone {REPO_URL} /content/BatchGeoCache
!pip install -e /content/BatchGeoCache

## 2. Imports

In [ ]:
# CONFIRM: import package name — assumed "batchgeocache" (lowercase of the repo name),
# matching Section 1's locked-in repo/package name "BatchGeoCache"
from batchgeocache.config import GeocoderConfig
from batchgeocache.geocoder import Geocoder
from batchgeocache.address_normalizer import AddressNormalizer
from batchgeocache.package_processor import PackageProcessor
from batchgeocache.state_manager import StateManager
from batchgeocache.logging_utils import PipelineLogger
from batchgeocache import io_utils
from batchgeocache.result_combiner import ResultCombiner
from batchgeocache.inspect_results import inspect_results

import pandas as pd
from pathlib import Path

## 3. Config — local Colab VM path only (never `/content/drive/...`)

In [ ]:
# All local, no Drive — per Section 1's locked-in decision.
PROJECT_ROOT = Path("/content/batchgeocache_run")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

# CONFIRM: GeocoderConfig field names (project_root / user_agent / package_size are guesses
# based on Decision Log entries re: generalized user_agent + 100-record package cap)
config = GeocoderConfig(
    project_root=PROJECT_ROOT,
    user_agent="batchgeocache-client-run",
    package_size=100,
)

## 4. Data in — client upload (no Drive)

In [ ]:
from google.colab import files

print("Upload the client's source address CSV:")
uploaded = files.upload()
source_filename = next(iter(uploaded))
source_path = PROJECT_ROOT / source_filename

with open(source_path, "wb") as f:
    f.write(uploaded[source_filename])

address_df = pd.read_csv(source_path)
address_df.head()

## 5. Processing loop — package, geocode, save, log, persist resume state

In [ ]:
logger = PipelineLogger(project_root=PROJECT_ROOT)
state_manager = StateManager(project_root=PROJECT_ROOT)
normalizer = AddressNormalizer(config=config)
geocoder = Geocoder(config=config)

# CONFIRM: PackageProcessor constructor args / method names
processor = PackageProcessor(
    config=config,
    normalizer=normalizer,
    geocoder=geocoder,
    logger=logger,
    state_manager=state_manager,
)

packages = processor.split_into_packages(address_df)  # up to 100 records each

for i, package_df in enumerate(packages):
    if state_manager.is_package_complete(i):
        logger.info(f"Skipping package {i} — already complete (resume)")
        continue

    result_df = processor.process_package(package_df, package_index=i)
    io_utils.save_package_outputs(result_df, package_index=i, project_root=PROJECT_ROOT)
    state_manager.mark_package_complete(i)
    logger.info(f"Package {i} complete")

## 6. Inspect results (geographic sanity check via `boundaries.toml`)

In [ ]:
# boundary_name defaults to "calgary" until more boundaries.toml entries exist (per Decision Log)
inspect_results(project_root=PROJECT_ROOT, boundary_name="calgary")

## 7. Combine + export + zip

In [ ]:
combiner = ResultCombiner(project_root=PROJECT_ROOT)
combined = combiner.combine_all()  # -> success / partial_success / failure
export_paths = combiner.export_combined(combined)
zip_path = io_utils.zip_combined_outputs(export_paths, project_root=PROJECT_ROOT)

print(f"Combined + zipped outputs at: {zip_path}")

## 8. Download combined zip to client

In [ ]:
files.download(str(zip_path))

## 9. Optional reset — purge local/temp files (destructive, opt-in)

In [ ]:
# This clears /content local files only — nothing in this notebook ever touches Drive.
CONFIRM_RESET = False  # flip to True to actually purge

if CONFIRM_RESET:
    io_utils.reset_project_files(project_root=PROJECT_ROOT)
    print("Local project files purged.")
else:
    print("Reset skipped — set CONFIRM_RESET = True to purge local files.")

---
**Drive-coupling self-check:** no cell above references `/content/drive/...` or
`google.colab.drive` — data in/out uses `google.colab.files.upload()` / `.download()` only,
per Section 1.